> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t05_schema_drift_demo` · notebook `04_schema_drift_demo` · started 2026-10-11 13:02 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `drift_mode=evolve`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 04 · Schema drift demo
Writes a synthetic batch into the landing volume that breaks the contract on purpose, then loads it:

| Line | Problem | Expected handling |
|---|---|---|
| 1 | new column `ai_assisted`, new nested field `owner.badge_counts` | column added to Bronze via `mergeSchema`; nested field kept in `_rescued_data` |
| 2 | `score` changes type: `"five"` instead of an integer | record quarantined |
| 3 | truncated JSON | record quarantined |
| 4 | no `question_id` | record quarantined |
| 5 | valid | loaded |

The batch finishes with status SUCCESS: 2 rows loaded, 3 quarantined. Nothing crashes.

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("drift_mode", "evolve", ["evolve", "rescue"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
import glob

from so_lakehouse.bronze import run_raw_to_bronze
from so_lakehouse.demo import DRIFT_BATCH_ID, make_drift_batch

cfg = build_config()
base = sorted(glob.glob(os.path.join(cfg.landing_root, "incremental", "incr_*", "questions.jsonl")))[0]
folder = make_drift_batch(cfg.landing_root, base)
audit = run_raw_to_bronze(spark, cfg, mode="backfill", landing_path=folder, fail_on_error=False)

Raw-to-Bronze: 1 batch(es) selected, mode=backfill
[SUCCESS] Raw-to-Bronze | questions                          | read=5 ins=2 upd=0 del=0 quar=3 | 25.1s | schema evolved: added ai_assisted; primary key constraint re-applied; rescued unknown fields: owner.badge_counts; quarantined 3: missing_primary_key=1, malforme
Raw-to-Bronze summary: {'run_id': 'raw_to_bronze-20261011T130239Z-8ec00d', 'steps': 1, 'by_status': {'SUCCESS': 1}, 'rows_inserted': 2, 'rows_updated': 0, 'rows_quarantined': 3}


In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs")).where(f"run_id = '{audit.run_id}'"))
display(spark.table(cfg.table("ops.schema_drift_events")).where(f"_batch_id = '{DRIFT_BATCH_ID}'"))
display(spark.table(cfg.table("bronze.quarantine_records")).where(f"_batch_id = '{DRIFT_BATCH_ID}'")
        .select("reason", "error_detail", "raw_record"))
display(spark.table(cfg.table("bronze.questions")).where(f"_batch_id = '{DRIFT_BATCH_ID}'")
        .select("question_id", "score", "_rescued_data", *([c for c in ["ai_assisted"]
                if c in spark.table(cfg.table("bronze.questions")).columns])))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
eb088d5e-0d96-4065-809f-d4b952e5464b,raw_to_bronze-20261011T130239Z-8ec00d,raw_to_bronze,Raw-to-Bronze,backfill,incremental,questions,incr_20261008T000000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261008T000000Z/questions.jsonl,workspace.so_bronze.questions,2026-10-11T13:02:40.851Z,2026-10-11T13:03:05.928Z,25.076457,SUCCESS,5,2,0,0,3,"schema evolved: added ai_assisted; primary key constraint re-applied; rescued unknown fields: owner.badge_counts; quarantined 3: missing_primary_key=1, malformed_json=1, type_mismatch=1",2026-10-11T13:03:05.928Z


event_id,run_id,_batch_id,entity,drift_type,column_path,record_count,example_value,action_taken,load_timestamp
1f5a3048febb346d5be13cb08ee61007ae518ace0fe37f04dcc67bc8e7f62f48,raw_to_bronze-20261011T130239Z-8ec00d,incr_20261008T000000Z,questions,new_column,owner.badge_counts,1,"{""gold"": 1, ""silver"": 4}",rescued,2026-10-11T13:02:59.175Z
b3e712c56603bc3ac4d38c757834c4821ddbcb721f603aa0df5fc2126b918bf3,raw_to_bronze-20261011T130239Z-8ec00d,incr_20261008T000000Z,questions,type_mismatch,score,1,"expected int, got str: ""five""",quarantined,2026-10-11T13:02:59.175Z
e2cbafa5ce79d907f0628eba761d5de08d0cef2d214271d7053b6dd34df0a8e3,raw_to_bronze-20261011T130239Z-8ec00d,incr_20261008T000000Z,questions,missing_primary_key,question_id,1,,quarantined,2026-10-11T13:02:59.175Z
86f86b92a1f71e2bfe16470b8bb1b8836563530e1f01fcb44778d36c7998b321,raw_to_bronze-20261011T130239Z-8ec00d,incr_20261008T000000Z,questions,new_column,ai_assisted,1,true,evolved,2026-10-11T13:02:59.175Z
86b8e16501b7c0aefb7dec1c6d10f13829de33ee875d499c6cf217b57cbf39b0,raw_to_bronze-20261011T130239Z-8ec00d,incr_20261008T000000Z,questions,malformed_json,,1,,quarantined,2026-10-11T13:02:59.175Z


reason              | error_detail                             | raw_record                              
--------------------+------------------------------------------+-----------------------------------------
malformed_json      | line is not a JSON object                | {"question_id": 990000003, "title":     
type_mismatch       | [{"path":"score","expected":"int","actua | {"tags": ["python", "pandas"], "owner": 
missing_primary_key | primary key (question_id) is null        | {"tags": ["sql", "spring-boot", "hsqldb"

question_id,score,_rescued_data,ai_assisted
990000001,4,"{""owner.badge_counts"":""{\""gold\"": 1, \""silver\"": 4}""}",true
990000004,1,,
